In [ ]:
!pip install -q soundfile speechbrain torchaudio
!pip install -q git+https://github.com/openai/whisper.git
!apt-get install -y ffmpeg

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 121.6/121.6 kB 4.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 788.2/788.2 kB 19.3 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 188.3/188.3 MB 6.2 MB/s eta 0:00:00
Reading package lists... Done
Building dependency tree... Done
Reading state information... Done
ffmpeg is already the newest version (7:4.4.2-0ubuntu0.22.04.1).
0 upgraded, 0 newly installed, 0 to remove and 2 not upgraded.


In [ ]:
import json
import whisper
import torch
import os
import ipywidgets as widgets

from speechbrain.inference.interfaces import foreign_class
from IPython.display import display, Audio
from google.colab.output import eval_js
from base64 import b64decode

In [28]:
import ipywidgets as widgets
from IPython.display import display
from google.colab.output import eval_js
from base64 import b64decode
import os
import time

# Create UI
record_button = widgets.Button(description="🎤 Start Recording", button_style='success')
status = widgets.Label("Click to record")

display(record_button, status)

def record_audio(_):
    duration = 7
    status.value = "🎤 Recording... Speak now!"

    js = f"""
    async function recordAudio() {{
      const stream = await navigator.mediaDevices.getUserMedia({{ audio: true }});
      const recorder = new MediaRecorder(stream);
      let chunks = [];

      recorder.ondataavailable = e => chunks.push(e.data);
      recorder.start();

      await new Promise(resolve => setTimeout(resolve, {duration * 1000}));

      recorder.stop();

      return new Promise(resolve => {{
        recorder.onstop = async () => {{
          const blob = new Blob(chunks);
          const arrayBuffer = await blob.arrayBuffer();
          const base64 = btoa(
            new Uint8Array(arrayBuffer)
              .reduce((data, byte) => data + String.fromCharCode(byte), '')
          );
          resolve(base64);
        }};
      }});
    }}
    recordAudio();
    """

    audio_base64 = eval_js(js)
    audio_bytes = b64decode(audio_base64)

    # 🔥 Remove old files (prevents reuse issue)
    if os.path.exists("recorded_audio.webm"):
        os.remove("recorded_audio.webm")
    if os.path.exists("recorded_audio.wav"):
        os.remove("recorded_audio.wav")

    # Save new file
    with open("recorded_audio.webm", "wb") as f:
        f.write(audio_bytes)

    status.value = "✅ Recording saved!"

# Attach function to button
record_button.on_click(record_audio)

Button(button_style='success', description='🎤 Start Recording', style=ButtonStyle())

Label(value='Click to record')

In [58]:
os.system("ffmpeg -i recorded_audio.webm -ac 1 -ar 16000 recorded_audio.wav")

audio_file = "recorded_audio.wav"

print("✅ Converted to WAV")

✅ Converted to WAV


In [35]:
print("📦 Loading models...")

whisper_model = whisper.load_model("base")

classifier = foreign_class(
    source="speechbrain/emotion-recognition-wav2vec2-IEMOCAP",
    pymodule_file="custom_interface.py",
    classname="CustomEncoderWav2vec2Classifier"
)

print("✅ Models loaded\n")

📦 Loading models...


INFO:speechbrain.utils.fetching:Fetch custom_interface.py: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch hyperparams.yaml: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached


Loading weights:   0%|          | 0/211 [00:00<?, ?it/s]

Wav2Vec2Model LOAD REPORT from: facebook/wav2vec2-base
Key                          | Status     |  | 
-----------------------------+------------+--+-
project_hid.weight           | UNEXPECTED |  | 
quantizer.weight_proj.bias   | UNEXPECTED |  | 
project_q.weight             | UNEXPECTED |  | 
project_q.bias               | UNEXPECTED |  | 
quantizer.codevectors        | UNEXPECTED |  | 
project_hid.bias             | UNEXPECTED |  | 
quantizer.weight_proj.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
INFO:speechbrain.utils.fetching:Fetch wav2vec2.ckpt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch model.ckpt: Fetching from HuggingFace Hub 'speechbrain/emotion-recognition-wav2vec2-IEMOCAP' if not cached
INFO:speechbrain.utils.fetching:Fetch label_encoder.txt: Fetching from HuggingFace Hub 'speechbrain

✅ Models loaded



In [36]:
def transcribe_audio(filename):
    print("📝 Transcribing...")
    result = whisper_model.transcribe(filename)
    print("✅ Done\n")
    return result["text"]

In [37]:
def detect_emotion(filename):
    print("😊 Detecting emotion...")

    out_prob, score, index, text_lab = classifier.classify_file(filename)

    probabilities = out_prob.squeeze().tolist()
    labels = classifier.hparams.label_encoder.ind2lab

    prob_dict = {
        labels[i]: float(probabilities[i])
        for i in range(len(labels))
    }

    print("✅ Done\n")

    return text_lab, float(score), prob_dict

In [59]:
if not os.path.exists("recorded_audio.wav"):
    print("❌ Please record audio first")
else:
    transcript = transcribe_audio("recorded_audio.wav")
    emotion, confidence, probs = detect_emotion("recorded_audio.wav")

    result = {
        "transcript": transcript,
        "predicted_emotion": emotion,
        "confidence_score": confidence,
        "emotion_probabilities": probs
    }

    print("\n=========== RESULT ===========\n")
    print("📝 Transcript:", transcript)
    print("😊 Emotion:", emotion)
    print("📊 Confidence:", round(confidence, 4))

    print("\n📈 Probabilities:")
    for k, v in probs.items():
        print(f"{k}: {v:.4f}")

    print("\n📦 JSON Output:")
    print(json.dumps(result, indent=4))

📝 Transcribing...


/usr/local/lib/python3.12/dist-packages/whisper/transcribe.py:132: UserWarning: FP16 is not supported on CPU; using FP32 instead
  warnings.warn("FP16 is not supported on CPU; using FP32 instead")


✅ Done

😊 Detecting emotion...
✅ Done


=========== RESULT ===========

📝 Transcript:  I don't like to do this. Good morning. Please leave me alone.
😊 Emotion: ['ang']
📊 Confidence: 1.0

📈 Probabilities:
neu: 0.0000
ang: 1.0000
hap: 0.0000
sad: 0.0000

📦 JSON Output:
{
    "transcript": " I don't like to do this. Good morning. Please leave me alone.",
    "predicted_emotion": [
        "ang"
    ],
    "confidence_score": 1.0,
    "emotion_probabilities": {
        "neu": 1.392764809732272e-12,
        "ang": 1.0,
        "hap": 6.123897916543442e-11,
        "sad": 2.216278376099279e-12
    }
}
